# Notebook 04 — Adaptation vs Fairness (mBRSET)
**BECITHCON 2026 · Fairness under domain shift · Experiment 4 of 4**

Notebook 03 showed that the portable-camera shift collapses overall accuracy **and** blows
open the image-quality fairness gap. This notebook asks the punchline question: **when we
adapt the model to recover accuracy on the portable camera, does the quality fairness gap
close, persist, or get worse?**

We compare three conditions on the **same held-out mBRSET test patients**:
1. **Zero-shot** — BRSET model, BRSET operating point (the notebook-03 baseline).
2. **Recalibration** — weights frozen; re-fit the operating threshold (and a global
   temperature) on an mBRSET adaptation split. The cheapest possible adaptation.
3. **Fine-tuning** — lightly fine-tune the model on the mBRSET adaptation split (supervising
   the two shared labels with weighted BCE), then re-fit the threshold.

For each we report **overall sensitivity/AUC** (did accuracy recover?) and the **image-quality
AUC and sensitivity gaps** (did fairness recover?). The interesting result is whichever way it
falls: if accuracy returns but the quality gap stays, the disparity is partly irreducible and
needs better image capture, not just model adaptation. If adaptation closes it, that is a
clean deployment recipe. Both are publishable.

**You fill in:** the mBRSET image directory, the checkpoint, and the mBRSET sex mapping.


In [ ]:
# ============================== CONFIG ==============================
MBRSET_LABELS    = "labels_mbrset.csv"
MBRSET_IMAGE_DIR = "/path/to/mbrset/images"      # <-- fill in
CHECKPOINT       = "/path/to/weighted_bce.pt"    # same BRSET model

# BRSET inputs, only used to recover the deployed (zero-shot) operating point
BRSET_LABELS = "labels_brset.csv"
SPLIT_FILE   = "split.csv"
BRSET_PREDS  = "fairness_outputs/brset_val_test_preds.csv"

MBRSET_SEX_MAP = {0: 'female', 1: 'male'}        # <-- CONFIRM (placeholder)
OUTPUT_DIR   = "fairness_outputs"

ADAPT_FRAC   = 0.70          # patient-level fraction used for adaptation; rest is test
DEVICE       = "cuda"
SEED         = 42
IMG_SIZE     = 224
BATCH_SIZE   = 64
NUM_WORKERS  = 4
TARGET_SENSITIVITY = 0.85

# light fine-tuning
FT_EPOCHS    = 8
FT_LR        = 1e-4
N_BOOTSTRAP  = 1000
ECE_BINS     = 10


In [ ]:
import os, numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")
np.random.seed(SEED)
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
torch.manual_seed(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)

LABELS_ORDER = ['diabetic_retinopathy','macular_edema','scar','nevus','amd',
                'vascular_occlusion','hypertensive_retinopathy','drusens','hemorrhage',
                'myopic_fundus','increased_cup_disc']
SHARED = ['diabetic_retinopathy','macular_edema']
SHARED_IDX = [LABELS_ORDER.index(l) for l in SHARED]
AXES = {'sex':'sex_group','quality':'quality_group','age':'age_group'}
dev = DEVICE if (DEVICE=='cpu' or torch.cuda.is_available()) else 'cpu'


## Section 1 — metrics, model, transforms

In [ ]:
def safe_auc(y,p):
    y=np.asarray(y); p=np.asarray(p)
    return roc_auc_score(y,p) if len(np.unique(y))>1 else np.nan
def ece(y,p,n_bins=ECE_BINS):
    y=np.asarray(y); p=np.asarray(p)
    if len(y)==0: return np.nan
    bins=np.linspace(0,1,n_bins+1); e=0.0
    for lo,hi in zip(bins[:-1],bins[1:]):
        m=(p>lo)&(p<=hi)
        if m.sum()==0: continue
        e+=m.mean()*abs(y[m].mean()-p[m].mean())
    return e
def sens_fpr(y,p,thr):
    y=np.asarray(y); pred=(np.asarray(p)>=thr).astype(int)
    tp=((pred==1)&(y==1)).sum(); fn=((pred==0)&(y==1)).sum()
    fp=((pred==1)&(y==0)).sum(); tn=((pred==0)&(y==0)).sum()
    return (tp/(tp+fn) if tp+fn else np.nan, fp/(fp+tn) if fp+tn else np.nan)
def thr_for_sens(y,p,target):
    y=np.asarray(y); p=np.asarray(p)
    if y.sum()==0: return 0.5
    chosen=np.unique(p).min()
    for t in np.unique(p)[::-1]:
        s,_=sens_fpr(y,p,t)
        if s>=target: chosen=t; break
    return float(chosen)

class BRSETClassifier(nn.Module):
    def __init__(self, n_labels):
        super().__init__()
        try: self.backbone = models.efficientnet_b0(weights=None)
        except TypeError: self.backbone = models.efficientnet_b0(pretrained=False)
        in_f = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Identity()
        self.head = nn.Linear(in_f, n_labels)
    def forward(self, x): return self.head(self.backbone(x))

_tf = transforms.Compose([transforms.Resize((IMG_SIZE,IMG_SIZE)), transforms.ToTensor(),
                          transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

class MBData(Dataset):
    def __init__(self, frame, d, with_labels=False):
        self.f=frame.reset_index(drop=True); self.d=d; self.wl=with_labels
    def __len__(self): return len(self.f)
    def _find(self, fn):
        for ext in ('', '.jpg', '.jpeg', '.png'):
            p=os.path.join(self.d, str(fn)+ext)
            if os.path.exists(p): return p
        raise FileNotFoundError(fn)
    def __getitem__(self,i):
        r=self.f.loc[i]; img=_tf(Image.open(self._find(r['file'])).convert('RGB'))
        if self.wl:
            y=np.array([r[l] if not pd.isna(r[l]) else -1 for l in SHARED], dtype='float32')
            return img, y
        return img, str(r['file'])


## Section 2 — load mBRSET, derive labels/subgroups, patient-level adapt/test split

In [ ]:
md = pd.read_csv(MBRSET_LABELS)
icdr = pd.to_numeric(md['final_icdr'], errors='coerce')
md['diabetic_retinopathy'] = (icdr>=1).astype('float'); md.loc[icdr.isna(),'diabetic_retinopathy']=np.nan
md['macular_edema'] = md['final_edema'].map({'yes':1.0,'no':0.0})
md['sex_group'] = md['sex'].map(MBRSET_SEX_MAP)
md['quality_group'] = md['final_quality'].map({'yes':'Adequate','no':'Inadequate'})
agenum = pd.to_numeric(md['age'], errors='coerce')
md['age_group'] = agenum.apply(lambda a: np.nan if pd.isna(a) else ('<40' if a<40 else ('40-59' if a<60 else '60+')))

rng=np.random.default_rng(SEED)
pts=md['patient'].unique().copy(); rng.shuffle(pts)
n_adapt=int(ADAPT_FRAC*len(pts)); adapt_pts=set(pts[:n_adapt])
md['phase']=np.where(md['patient'].isin(adapt_pts),'adapt','test')
adapt=md[md.phase=='adapt'].copy(); test=md[md.phase=='test'].copy()
print("adapt patients:", md[md.phase=='adapt']['patient'].nunique(),
      "| test patients:", md[md.phase=='test']['patient'].nunique())
print("test images:", len(test))


## Section 3 — inference helper and a reusable evaluator

In [ ]:
@torch.no_grad()
def predict(frame, net):
    net.eval()
    loader=DataLoader(MBData(frame, MBRSET_IMAGE_DIR), batch_size=BATCH_SIZE,
                      shuffle=False, num_workers=NUM_WORKERS)
    files,probs=[],[]
    for x,fn in loader:
        probs.append(torch.sigmoid(net(x.to(dev))).cpu().numpy()); files.extend(fn)
    P=np.concatenate(probs,0)
    out=pd.DataFrame({'file':files})
    for l,j in zip(SHARED, SHARED_IDX): out['prob_'+l]=P[:,j]
    return out

def evaluate(frame_with_probs, thresholds):
    # returns overall metrics + quality/sex/age gaps for each shared label
    overall=[]; gaps=[]
    for l in SHARED:
        sub=frame_with_probs[frame_with_probs[l].notna()]
        y=sub[l].values; p=sub['prob_'+l].values
        s,f=sens_fpr(y,p,thresholds[l])
        overall.append(dict(label=l, auc=safe_auc(y,p), sens=s, fpr=f, ece=ece(y,p)))
        for axis,col in AXES.items():
            vals_auc=[]; vals_sens=[]
            for g,gdf in sub.dropna(subset=[col]).groupby(col):
                yy=gdf[l].values; pp=gdf['prob_'+l].values
                vals_auc.append(safe_auc(yy,pp))
                ss,_=sens_fpr(yy,pp,thresholds[l]); vals_sens.append(ss)
            va=[v for v in vals_auc if not np.isnan(v)]; vs=[v for v in vals_sens if not np.isnan(v)]
            gaps.append(dict(label=l, axis=axis,
                             auc_gap=(max(va)-min(va)) if len(va)>=2 else np.nan,
                             sens_gap=(max(vs)-min(vs)) if len(vs)>=2 else np.nan))
    return pd.DataFrame(overall), pd.DataFrame(gaps)


## Section 4 — Condition 1: zero-shot

In [ ]:
net = BRSETClassifier(len(LABELS_ORDER)).to(dev)
ck = torch.load(CHECKPOINT, map_location=dev)
st = ck.get('model_state_dict', ck) if isinstance(ck,dict) else ck
net.load_state_dict(st)

# zero-shot operating point = BRSET validation threshold (the deployed point from notebook 03)
bd = pd.read_csv(BRSET_LABELS).merge(pd.read_csv(SPLIT_FILE)[['patient_id','split']],
                                     on='patient_id', how='left')
bd = bd.merge(pd.read_csv(BRSET_PREDS), on='image_id', how='inner')
bval = bd[bd.split=='val']
THR_ZS = {l: thr_for_sens(bval[l], bval['prob_'+l], TARGET_SENSITIVITY) for l in SHARED}
print("Zero-shot (BRSET) thresholds:", {k: round(v,3) for k,v in THR_ZS.items()})

# frozen-model predictions on the adapt split (reused to fit the recalibration threshold)
adapt_pred0 = predict(adapt, net)
adapt0 = adapt.merge(adapt_pred0, on='file')

test_pred0 = predict(test, net)
test0 = test.merge(test_pred0, on='file')
ov_zs, gap_zs = evaluate(test0, THR_ZS)
print("ZERO-SHOT overall:\n", ov_zs.round(3).to_string(index=False))
print("\nZERO-SHOT gaps:\n", gap_zs.round(3).to_string(index=False))


## Section 5 — Condition 2: recalibration (weights frozen)

Re-fit the operating threshold on the adaptation split. This restores the operating point on
the portable camera without changing a single weight, the cheapest adaptation available.


In [ ]:
# threshold re-fit on the mBRSET adapt split (weights frozen, so AUC is unchanged vs zero-shot;
# only the operating point moves, which is exactly what recalibration can and cannot fix)
THR_RECAL = {l: thr_for_sens(adapt0[adapt0[l].notna()][l],
                             adapt0[adapt0[l].notna()]['prob_'+l], TARGET_SENSITIVITY) for l in SHARED}
ov_rc, gap_rc = evaluate(test0, THR_RECAL)
print("RECALIBRATION overall:\n", ov_rc.round(3).to_string(index=False))
print("\nRECALIBRATION gaps:\n", gap_rc.round(3).to_string(index=False))


## Section 6 — Condition 3: light fine-tuning on the adaptation split

We fine-tune on the adaptation patients, supervising only the two shared labels with weighted
BCE (other outputs receive no gradient). Low learning rate, few epochs, to represent a
realistic lightweight adaptation rather than full retraining.


In [ ]:
# pos_weight per shared label from adapt split
pw=[]
for l in SHARED:
    yv=adapt[adapt[l].notna()][l].values
    pw.append((len(yv)-yv.sum())/max(yv.sum(),1))
pos_weight=torch.tensor(pw, dtype=torch.float32, device=dev)

ft = BRSETClassifier(len(LABELS_ORDER)).to(dev); ft.load_state_dict(st)
opt=torch.optim.Adam(ft.parameters(), lr=FT_LR)
loader=DataLoader(MBData(adapt, MBRSET_IMAGE_DIR, with_labels=True),
                  batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)

ft.train()
for ep in range(FT_EPOCHS):
    tot=0.0
    for x,y in loader:
        x=x.to(dev); y=y.to(dev)             # y: (B,2), -1 marks missing label
        logits=ft(x)[:, SHARED_IDX]
        mask=(y>=0).float()
        bce=nn.functional.binary_cross_entropy_with_logits(
                logits, torch.clamp(y,0,1), pos_weight=pos_weight, reduction='none')
        loss=(bce*mask).sum()/mask.sum().clamp(min=1)
        opt.zero_grad(); loss.backward(); opt.step(); tot+=loss.item()
    print(f"epoch {ep+1}/{FT_EPOCHS}  loss={tot/len(loader):.4f}")

adapt_predF = predict(adapt, ft); adaptF = adapt.merge(adapt_predF, on='file')
THR_FT={l: thr_for_sens(adaptF[adaptF[l].notna()][l],
                        adaptF[adaptF[l].notna()]['prob_'+l], TARGET_SENSITIVITY) for l in SHARED}
test_predF=predict(test, ft); testF=test.merge(test_predF, on='file')
ov_ft, gap_ft = evaluate(testF, THR_FT)
print("\nFINE-TUNED overall:\n", ov_ft.round(3).to_string(index=False))
print("\nFINE-TUNED gaps:\n", gap_ft.round(3).to_string(index=False))


## Section 7 — The punchline: did accuracy recover, did fairness recover?

In [ ]:
def tag(ov, gap, name):
    ov=ov.copy(); ov['condition']=name
    q=gap[gap.axis=='quality'][['label','auc_gap','sens_gap']].rename(
        columns={'auc_gap':'quality_auc_gap','sens_gap':'quality_sens_gap'})
    return ov.merge(q, on='label')

summary=pd.concat([tag(ov_zs,gap_zs,'zero_shot'),
                   tag(ov_rc,gap_rc,'recalibration'),
                   tag(ov_ft,gap_ft,'fine_tuned')], ignore_index=True)
summary=summary[['condition','label','auc','sens','fpr','quality_auc_gap','quality_sens_gap']]
print("Accuracy vs quality-fairness across adaptation conditions:")
print(summary.round(3).to_string(index=False))
summary.to_csv(os.path.join(OUTPUT_DIR,'adaptation_summary.csv'), index=False)


## Section 8 — Figure: accuracy recovered vs quality gap

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,4))
order=['zero_shot','recalibration','fine_tuned']
for ax,l in zip(axes, SHARED):
    s=summary[summary.label==l].set_index('condition').reindex(order)
    x=np.arange(len(order)); w=0.35
    ax.bar(x-w/2, s['sens'], w, label='overall sensitivity')
    ax.bar(x+w/2, s['quality_auc_gap'], w, label='quality AUC gap')
    ax.set_xticks(x); ax.set_xticklabels(order, rotation=20); ax.set_title(l)
axes[0].legend()
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR,'figF_adaptation.png'),dpi=200); plt.close()
print("Saved figF_adaptation.png and adaptation_summary.csv to", OUTPUT_DIR)


## Section 9 — Reading it (for the paper)

- **Compare `sens` across conditions:** recalibration and fine-tuning should lift overall
  sensitivity back up from the zero-shot collapse. That confirms adaptation restores accuracy.
- **Key structural point:** recalibration freezes the weights, so it cannot change AUC or the
  AUC gap at all, it only moves the operating point. So `quality_auc_gap` will be identical
  for zero-shot and recalibration, and only fine-tuning can move it. That makes the AUC gap a
  clean test of whether changing the features (not just the threshold) repairs ranking fairness.
- **Now compare `quality_auc_gap` for zero-shot vs fine-tuned.** This is the whole point.
  - If the gap **stays high** while sensitivity recovers, the message is sharp: adaptation
    buys back accuracy but not fairness, because inadequate images lack the signal, so the
    fix has to be at image capture (quality gating, recapture prompts), not just the model.
  - If the gap **shrinks**, you have a concrete deployment recipe: a small labelled
    adaptation set restores both accuracy and quality-fairness on the portable camera.
- Either way, pair the claim with the bootstrap CIs (rerun the notebook-03 bootstrap on the
  fine-tuned test predictions if you want intervals on the final gap).
- **Caveat to state:** fine-tuning supervised only the two shared labels and used a small
  adaptation split, so this is "lightweight adaptation," not full retraining; say so.

This closes the four-experiment arc. Next step is drafting: results first, in the order
01 -> 02 -> 03 -> 04, then methods, intro, discussion. Send me the Section 7 summary table
and I will start the results section.
